# 04-3. 완전 중복 제거 및 누수 방지 그룹 분할

모델이 사용하는 document+summary 완전 중복을 하나만 남기고, 같은 document 또는 같은 summary를 공유하는 행을 하나의 연결 그룹으로 묶는다. test는 고정하고 test 그룹과 겹치는 train/validation 행은 제외한다.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT=Path(r"C:\Users\KDS-30\Desktop\Projects\06_Team_Project4")
assert Path(sys.prefix).resolve()==(PROJECT/".venv").resolve(), "커널을 Python 3.11 (Team4 News GPU)로 바꾸세요."
INPUT=PROJECT/"processed_data"/"04-2_cleaned"
OUTPUT=PROJECT/"processed_data"/"04-3_final"
RESULTS=PROJECT/"review_data"/"04-3_dedup_group_split"
assert INPUT.exists(), "먼저 04-2를 실행하세요."
OUTPUT.mkdir(parents=True,exist_ok=True); RESULTS.mkdir(parents=True,exist_ok=True)


In [2]:
splits=["train","validation","test"]
df=pd.concat([pd.read_csv(INPUT/f"{s}.csv") for s in splits],ignore_index=True)
assert df.row_id.is_unique
priority={"test":0,"validation":1,"train":2}
ordered=df.assign(_priority=df.split.map(priority),_position=np.arange(len(df))).sort_values(["_priority","source_index","_position"])
ordered["keep_exact"]=~ordered.duplicated(["document","summary"],keep="first")
df=df.merge(ordered[["row_id","keep_exact"]],on="row_id",validate="one_to_one")
exact_excluded=df[~df.keep_exact].copy(); work=df[df.keep_exact].copy().reset_index(drop=True)

# 같은 document 또는 같은 summary를 공유하면 같은 연결 요소로 묶습니다.
parent=list(range(len(work)))
def find(x):
    while parent[x]!=x:
        parent[x]=parent[parent[x]]; x=parent[x]
    return x
def union(a,b):
    ra,rb=find(a),find(b)
    if ra!=rb: parent[rb]=ra
for column in ["document","summary"]:
    for indices in work.groupby(column,sort=False).indices.values():
        indices=list(indices)
        for index in indices[1:]: union(indices[0],index)
work["group_id"]=[find(i) for i in range(len(work))]
group_splits=work.groupby("group_id").split.agg(set)
def destination(values):
    if "test" in values: return "test"
    if "validation" in values: return "validation"
    return "train"
destinations=group_splits.map(destination)
work["assigned_split"]=work.group_id.map(destinations)
work["keep_group"]=~((work.assigned_split=="test")&(work.split!="test"))
work["split_action"]="kept"
work.loc[~work.keep_group,"split_action"]="excluded_test_overlap"
moved=work.keep_group & work.split.ne(work.assigned_split)
work.loc[moved,"split_action"]="moved_to_"+work.loc[moved,"assigned_split"]


In [3]:
saved=[]
for split in splits:
    output=work[work.keep_group & work.assigned_split.eq(split)].copy(); output.to_csv(OUTPUT/f"{split}.csv",index=False,encoding="utf-8-sig"); saved.append(output)
final=pd.concat(saved,ignore_index=True)
assert final.duplicated(["document","summary"]).sum()==0
assert final.groupby("document").assigned_split.nunique().max()==1
assert final.groupby("summary").assigned_split.nunique().max()==1
assert len(final[final.split.eq("test")])==len(work[work.split.eq("test")])
exact_excluded[["row_id","split","title","link","document","summary"]].to_csv(RESULTS/"excluded_exact_duplicates.csv",index=False,encoding="utf-8-sig")
work[["row_id","split","assigned_split","group_id","split_action","title","link"]].to_csv(RESULTS/"group_split_manifest.csv",index=False,encoding="utf-8-sig")
summary=pd.DataFrame({"split":splits,"input_rows":[int((df.split==s).sum()) for s in splits],"exact_duplicates_excluded":[int(((df.split==s)&~df.keep_exact).sum()) for s in splits],"final_rows":[int((final.assigned_split==s).sum()) for s in splits],"test_overlap_excluded":[int(((work.split==s)&~work.keep_group).sum()) for s in splits],"moved_in":[int(((final.assigned_split==s)&(final.split!=s)).sum()) for s in splits]})
summary.to_csv(RESULTS/"split_summary.csv",index=False,encoding="utf-8-sig")
pd.DataFrame([{"input_data":str(INPUT),"output_data":str(OUTPUT),"exact_key":"cleaned document + summary","group_rule":"connected component of exact document OR exact summary","test_policy":"original test fixed; overlapping non-test rows excluded"}]).to_csv(RESULTS/"run_info.csv",index=False,encoding="utf-8-sig")
display(summary)

,split,input_rows,exact_duplicates_excluded,final_rows,test_overlap_excluded,moved_in
0,train,21073,1831,19236,4,0
1,validation,2345,106,2240,1,2
2,test,2606,42,2564,0,0
